In [1]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
gpt-5-nano
랭스미스로 추적 중입니다 : lsv2_pt_ef


In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment=os.environ["MODEL_NAME"],
    azure_endpoint=os.environ["END_POINT"],
    openai_api_version="2025-03-01-preview",
    openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

In [3]:
import subprocess

WIKI_PORT = 8001
WEATHER_PORT = 8002
OVERPASS_PORT = 8003
try:
    subprocess.run(["fuser", "-k", f"{WIKI_PORT}/tcp"], capture_output=True)
    subprocess.run(["fuser", "-k", f"{WEATHER_PORT}/tcp"], capture_output=True)
    subprocess.run(["fuser", "-k", f"{OVERPASS_PORT}/tcp"], capture_output=True)
except Exception:
    pass

MCP_URL = os.getenv("MCP_URL", f"http://localhost:{WIKI_PORT}/mcp")
MCP_URL = os.getenv("MCP_URL", f"http://localhost:{WEATHER_PORT}/mcp")
MCP_URL = os.getenv("MCP_URL", f"http://localhost:{OVERPASS_PORT}/mcp")

import sys, subprocess
print("using python:", sys.executable)
subprocess.Popen([sys.executable, "geocode_server.py"])
subprocess.Popen([sys.executable, "wiki_server.py", "--port", str(WIKI_PORT)])
subprocess.Popen([sys.executable, "weather_server.py", "--port", str(WEATHER_PORT)])
subprocess.Popen([sys.executable, "overpass_server.py", "--port", str(OVERPASS_PORT)])


using python: /anaconda/envs/kt_env/bin/python


<Popen: returncode: None args: ['/anaconda/envs/kt_env/bin/python', 'overpas...>

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import sys
client = MultiServerMCPClient(  
    {
        "trip-wiki": {
            "transport": "streamable_http",
            "url": f"http://localhost:{WIKI_PORT}/mcp",
        },
        "geocode": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["./geocode_server.py"],
        },
        "trip-weather": {
            "transport": "streamable_http",
            "url": f"http://localhost:{WEATHER_PORT}/mcp",
        },
        "overpass": {
            "transport": "streamable_http",
            "url": f"http://localhost:{OVERPASS_PORT}/mcp",
        }
    }
)

In [7]:
tools = await client.get_tools()
tools

INFO:     127.0.0.1:43646 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:41934 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55162 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55182 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43654 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41946 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55168 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41940 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:43658 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43666 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:41952 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55188 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55194 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:43682 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:41956 - "DELETE /mcp HTTP/1.1" 200 OK


[StructuredTool(name='wiki_info', description='목적지의 위키요약과 관광 POI 후보 수집.\n# input:\n destination : 지역명을 영어 단어로 입력\n# output:\n wiki_summary : 위키요약\n pois : JSON 형식의 관광 POI 후보\nREST v1 summary → 실패 시 검색 API fallback.', args_schema={'additionalProperties': False, 'properties': {'destination': {'type': 'string'}}, 'required': ['destination'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7aedcd6754e0>),
 StructuredTool(name='get_geocode', description='위도/경도/타임존 조회. \ninput : 지역명을 영어 단어로 입력\noutput : 위도, 경도, 타임존\n내부적으로 Open-Meteo Geocoding API 사용.', args_schema={'additionalProperties': False, 'properties': {'destination': {'type': 'string'}}, 'required': ['destination'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7

In [8]:
wiki_text = await tools[0].ainvoke({"destination":"Tokyo"})
print(wiki_text)

INFO:     127.0.0.1:39654 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39664 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:39670 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39684 - "POST /mcp HTTP/1.1" 200 OK
destination: Tokyo
WIKI_SUMMARY: https://en.wikipedia.org/api/rest_v1/page/summary/Tokyo
INFO:     127.0.0.1:39688 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39696 - "DELETE /mcp HTTP/1.1" 200 OK
[{'type': 'text', 'text': '{"wiki_summary":"Tokyo, officially the Tokyo Metropolis, is the capital and most populous city of Japan. With a population of over 14 million in the city proper in 2023, it is one of the most populous urban areas in the world. The Greater Tokyo Area, which includes Tokyo and parts of six neighboring prefectures, is the most populous metropolitan area in the world, with 41 million residents as of 2024.","pois":[{"title":"Lists of tourist attractions","snippet":"are lists of <span class=\\"searchmatch\\">tourist</span> <span class=\\"sea

In [9]:
coord = await tools[1].ainvoke({"destination":"Tokyo"})
coord

[{'type': 'text',
  'text': '{"lat":35.6895,"lon":139.69171,"tz":"Asia/Tokyo"}',
  'id': 'lc_da32a0d8-0163-4a58-92a8-5177fea2faba'}]

In [10]:
inp = {"lat": 35.681236, "lon": 139.767125, "tz": "Asia/Tokyo", "start_date": "2026-02-12", "nights": 3}
await tools[2].ainvoke(inp)

INFO:     127.0.0.1:46006 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46020 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46024 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:46028 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46044 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46060 - "DELETE /mcp HTTP/1.1" 200 OK


[{'type': 'text',
  'text': '{"latitude":35.7,"longitude":139.75,"generationtime_ms":0.09119510650634766,"utc_offset_seconds":32400,"timezone":"Asia/Tokyo","timezone_abbreviation":"GMT+9","elevation":10.0,"hourly_units":{"time":"iso8601","temperature_2m":"°C","precipitation":"mm","windspeed_10m":"km/h"},"hourly":{"time":["2026-02-26T00:00","2026-02-26T01:00","2026-02-26T02:00","2026-02-26T03:00","2026-02-26T04:00","2026-02-26T05:00","2026-02-26T06:00","2026-02-26T07:00","2026-02-26T08:00","2026-02-26T09:00","2026-02-26T10:00","2026-02-26T11:00","2026-02-26T12:00","2026-02-26T13:00","2026-02-26T14:00","2026-02-26T15:00","2026-02-26T16:00","2026-02-26T17:00","2026-02-26T18:00","2026-02-26T19:00","2026-02-26T20:00","2026-02-26T21:00","2026-02-26T22:00","2026-02-26T23:00"],"temperature_2m":[5.7,5.4,5.2,5.6,5.6,5.5,5.9,5.8,6.2,7.1,8.2,9.2,10.1,10.8,11.3,11.5,11.2,10.3,9.5,8.8,8.3,7.8,7.4,7.2],"precipitation":[0.2,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.

In [11]:
inp = {"lat": 35.681236, "lon": 139.767125, "categories": ["hotel","cafe","restaurant"], "radius_m": 1000, "limit": 30}
await tools[3].ainvoke({"input": inp})

INFO:     127.0.0.1:36296 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:36314 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:36300 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:36326 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37316 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37320 - "DELETE /mcp HTTP/1.1" 200 OK


[{'type': 'text',
  'text': '{"places":[{"name":"マンダリンオリエンタル","category":"hotel","address":"","lat":35.687145,"lon":139.772888,"stars":"5","opening_hours":null,"phone":null,"website":null},{"name":"サンマルクカフェ","category":"cafe","address":"11, 中央区","lat":35.6727334,"lon":139.7673145,"stars":null,"opening_hours":"Mo-Fr 07:00-22:30; Sa,Su,PH 07:30-22:30","phone":"+81 3-3535-0309","website":"http://www.saint-marc-hd.com/cafe/"},{"name":"ドトールコーヒーショップ","category":"cafe","address":"11, 中央区","lat":35.6727804,"lon":139.767244,"stars":null,"opening_hours":"Mo-Sa 07:30-22:00; Su,PH 08:00-22:00","phone":"+81 3-3567-9003","website":"https://www.doutor.co.jp/dcs/"},{"name":"ドトールコーヒーショップ","category":"cafe","address":"12, 中央区","lat":35.6754844,"lon":139.7695215,"stars":null,"opening_hours":"Mo-Fr 06:45-21:00; Sa 08:00-19:30; Su,PH 08:00-19:00","phone":"+81 3-5250-7780","website":"https://www.doutor.co.jp/dcs/"},{"name":"主水","category":"restaurant","address":"","lat":35.6857074,"lon":139.7743952,"stars":

In [12]:
from langchain_core.tools import tool

@tool
async def tool_list(_:None = None) -> str:
    """wiki 툴 목록을 반환합니다. 입력값은 None입니다."""
    return tools

from datetime import datetime
@tool
async def get_date(_:None = None) -> str:
    """오늘 날짜를 반환합니다. 입력값은 None입니다."""
    return datetime.now().strftime("%Y-%m-%d")

@tool
async def wiki_poi_search(text: str) -> str:
    """전체 위키 제목/스니펫을 바탕으로 관광 명소(indoor/outdoor/theme)를 분류 및 중복 제거.
    #input 
     text : 관광 명소가 포함된 일반 텍스트
    #output 
     JSON 형식의 관광 명소 분류 및 중복 제거한 리스트"""
    prompt = await client.get_prompt("trip-wiki", "poi_search_prompt", arguments = {"text" : text})

    result = await llm.ainvoke(prompt)
    return result.content


tools += [tool_list, get_date, wiki_poi_search]
tools

[StructuredTool(name='wiki_info', description='목적지의 위키요약과 관광 POI 후보 수집.\n# input:\n destination : 지역명을 영어 단어로 입력\n# output:\n wiki_summary : 위키요약\n pois : JSON 형식의 관광 POI 후보\nREST v1 summary → 실패 시 검색 API fallback.', args_schema={'additionalProperties': False, 'properties': {'destination': {'type': 'string'}}, 'required': ['destination'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7aedcd6754e0>),
 StructuredTool(name='get_geocode', description='위도/경도/타임존 조회. \ninput : 지역명을 영어 단어로 입력\noutput : 위도, 경도, 타임존\n내부적으로 Open-Meteo Geocoding API 사용.', args_schema={'additionalProperties': False, 'properties': {'destination': {'type': 'string'}}, 'required': ['destination'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7

In [13]:
await tool_list.ainvoke(None)

[StructuredTool(name='wiki_info', description='목적지의 위키요약과 관광 POI 후보 수집.\n# input:\n destination : 지역명을 영어 단어로 입력\n# output:\n wiki_summary : 위키요약\n pois : JSON 형식의 관광 POI 후보\nREST v1 summary → 실패 시 검색 API fallback.', args_schema={'additionalProperties': False, 'properties': {'destination': {'type': 'string'}}, 'required': ['destination'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7aedcd6754e0>),
 StructuredTool(name='get_geocode', description='위도/경도/타임존 조회. \ninput : 지역명을 영어 단어로 입력\noutput : 위도, 경도, 타임존\n내부적으로 Open-Meteo Geocoding API 사용.', args_schema={'additionalProperties': False, 'properties': {'destination': {'type': 'string'}}, 'required': ['destination'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7

In [14]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

SYSTEM_PROMPT = """
너는 여행의 다양한 정보를 추출해서 사용자에게 제시하는 도우미 에이전트다.
#도구 사용 원칙:
1) '날씨(weather_forecast)' 또는 '장소/호텔/카페/식당(find_places)'를 요청받으면
   - 목적지의 위도/경도/타임존이 없다면 반드시 먼저 get_geocode 호출해 확보한다.
   - 그 다음 거기에서 확보한 정보로 weather_forecast 또는 find_places 호출한다.
2) 사용자가 목적지를 바꾸면 새로 get_geocode를 갱신한다. geocode에는 지역의 이름을 영어로 입력한다.
3) 답변에는 핵심 정보만 간결히 요약하고, 필요한 경우 선택한 파라미터(날짜/반경/카테고리)도 함께 제시한다.

#도구 :
get_geocode - 목적지 위도/경도/타임존 확보 
get_weather - 날씨 정보 확보 / 입력 예시 "lat": 35.681236, "lon": 139.767125, "tz": "Asia/Tokyo", "start_date": "2025-11-01", "nights": 3
get_places - 장소 정보 확보 / 입력 예시 "lat": 35.681236, "lon": 139.767125, "categories": ["hotel","cafe","restaurant"], "radius_m": 1000, "limit": 30
tool_list - 도구 목록 확보 / 입력 None
wiki_info - 목적지의 위키요약과 관광 POI 후보 수집
wiki_poi_search - text를 입력으로 받아서 관광 명소 분류
"""

agent = create_agent(
    llm,
    tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    debug=False
)

In [15]:
async def run_agent(query, user_id, mode = 'invoke'):
    if mode == 'invoke':
        result =  await agent.ainvoke({"messages": [{"role": "user", "content": query}]}, config={"configurable": {"thread_id": user_id}})
        return result['messages'][-1].content
    else:
        async for chunk in agent.astream(
            {"messages": [{"role": "user", "content": query}]},
            config={"configurable": {"thread_id": user_id}},
            stream_mode="values",
        ):
            msgs = chunk.get("messages")
            if msgs:
                print(msgs[-1].pretty_repr(html=True))


In [16]:
await run_agent('피렌체의 위치는?', '1', mode = 'stream')

================================ Human Message =================================

피렌체의 위치는?
================================== Ai Message ==================================
Tool Calls:
  get_geocode (call_MG2d91NCKzWEXmZ9QzNbRDCy)
 Call ID: call_MG2d91NCKzWEXmZ9QzNbRDCy
  Args:
    destination: Florence
================================= Tool Message =================================
Name: get_geocode

[{'type': 'text', 'text': '{"lat":43.77925,"lon":11.24626,"tz":"Europe/Rome"}', 'id': 'lc_fa412c2f-2195-493c-9ba4-5b58f8afaed6'}]
================================== Ai Message ==================================

피렌체(Florence, 이탈리아)의 위치
- 위도: 43.77925
- 경도: 11.24626
- 타임존: Europe/Rome

원하시면 이 좌표를 기준으로 날씨 예보나 주변 장소를 검색해 드릴게요. 필요한 파라미터(날짜, 반경, 카테고리)를 알려 주세요.


In [17]:
await run_agent('거기 유명한 관광명소가 뭐야?', '1', mode = 'stream')

================================ Human Message =================================

거기 유명한 관광명소가 뭐야?
================================== Ai Message ==================================
Tool Calls:
  wiki_info (call_yZLeHZ15Ht9py1KnQGkknocm)
 Call ID: call_yZLeHZ15Ht9py1KnQGkknocm
  Args:
    destination: Florence
INFO:     127.0.0.1:51222 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:51246 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:51234 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:51260 - "POST /mcp HTTP/1.1" 200 OK
destination: Florence
WIKI_SUMMARY: https://en.wikipedia.org/api/rest_v1/page/summary/Florence
INFO:     127.0.0.1:51274 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:51278 - "DELETE /mcp HTTP/1.1" 200 OK
================================= Tool Message =================================
Name: wiki_info

[{'type': 'text', 'text': '{"wiki_summary":"Florence is the capital and most populous city of the Italian region of Tuscany, with 362,353 inhabitants as o

In [18]:
await run_agent('어떤 도구를 가지고 있어?', '1', mode = 'stream')

================================ Human Message =================================

어떤 도구를 가지고 있어?
================================== Ai Message ==================================
Tool Calls:
  tool_list (call_8vXu1ycHaMf7uLNIlsDgOLZR)
 Call ID: call_8vXu1ycHaMf7uLNIlsDgOLZR
  Args:
================================= Tool Message =================================
Name: tool_list

[StructuredTool(name='wiki_info', description='목적지의 위키요약과 관광 POI 후보 수집.\n# input:\n destination : 지역명을 영어 단어로 입력\n# output:\n wiki_summary : 위키요약\n pois : JSON 형식의 관광 POI 후보\nREST v1 summary → 실패 시 검색 API fallback.', args_schema={'additionalProperties': False, 'properties': {'destination': {'type': 'string'}}, 'required': ['destination'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7aedcd6754e0>), StructuredTool(name='get_geocode', description='위도/경도/타임존 조회. \ninput : 지역명을 영

In [19]:
await run_agent('오늘 날짜는?', '1', mode = 'stream')

================================ Human Message =================================

오늘 날짜는?
================================== Ai Message ==================================
Tool Calls:
  get_date (call_VPuhhDATpIRi1YG9Pli11Wf7)
 Call ID: call_VPuhhDATpIRi1YG9Pli11Wf7
  Args:
================================= Tool Message =================================
Name: get_date

2026-02-26
================================== Ai Message ==================================

오늘 날짜는 2026-02-26입니다. 필요하신 경우 요일도 알려드릴게요.


In [20]:
await run_agent('내일 피렌체의 날씨는 어때?', '1', mode = 'stream')

================================ Human Message =================================

내일 피렌체의 날씨는 어때?
================================== Ai Message ==================================
Tool Calls:
  weather_forecast (call_ka2Kia9C7GKN8IMeQz5BhybN)
 Call ID: call_ka2Kia9C7GKN8IMeQz5BhybN
  Args:
    lat: 43.77925
    lon: 11.24626
    tz: Europe/Rome
    start_date: 2026-02-27
    nights: 1
INFO:     127.0.0.1:37906 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37922 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37938 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:37954 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:34996 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35006 - "DELETE /mcp HTTP/1.1" 200 OK
================================= Tool Message =================================
Name: weather_forecast

[{'type': 'text', 'text': '{"latitude":43.78,"longitude":11.24,"generationtime_ms":0.12421607971191406,"utc_offset_seconds":3600,"timezone":"Europe/Rome","timezone_a

In [22]:
await run_agent('피렌체의 관광명소를 wiki_poi_search의 출력 형태로 바로 출력해줘', '2', mode = 'stream')

================================ Human Message =================================

피렌체의 관광명소를 wiki_poi_search의 출력 형태로 바로 출력해줘


BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_82CcLSkZdwKybzb9j1Gckfpu", 'type': 'invalid_request_error', 'param': 'messages.[3].role', 'code': None}}